# ***SATRIA DATA COMPETITION***

Video Clasification

# Summarize Transcribe

In [1]:
!pip install transformers accelerate bitsandbytes sentencepiece

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.1/60.1 MB 28.1 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.7 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 105.0 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 81.4 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 35.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.6 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 5.2 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 30.9 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 13.6 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 2.0 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 82.3 MB/s eta 0:00:00:00:0100:01
  Attempting un

In [4]:
# ============================
# 1) IMPORTS & CONFIG
# ============================
import os, re, gc
import pandas as pd
from tqdm.auto import tqdm
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
import warnings
warnings.filterwarnings('ignore')

# ---- Konfigurasi file & kolom ----
INPUT_CSV   = "/kaggle/input/transcript/transcriptions.csv"                 # ganti ke path CSV input kamu
OUTPUT_CSV  = "data_with_summary.csv"    # hasil akan disimpan/diupdate di sini
TEXT_COL    = "text"
SUMMARY_COL = "summary_2para"

# ---- Model & decoding config ----
MODEL_ID = "Qwen/Qwen2.5-7B-Instruct"   # 7B Instruct; aman di T4/L4 dgn 4-bit
MAX_CTX  = 8192                         # panjang konteks aman
MAP_MAX_NEW    = 160                    # target 5–7 kalimat
REDUCE_MAX_NEW = 240                    # target total 2 paragraf

# stabil & konsisten (cocok untuk clustering downstream)
GEN_KW = dict(
    do_sample=False,
    temperature=0.2,
    top_p=0.9,
    repetition_penalty=1.05,
)

# ---- Chunking config ----
CHUNK_TOKENS     = 6000     # ~6k token/segmen
OVERLAP_TOKENS   = 250      # overlap kecil untuk kontinuitas
FALLBACK_CHUNK   = 4500     # dipakai otomatis kalau OOM

# ============================
# 2) LOAD MODEL (4-bit)
# ============================
bnb = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
)

tok = AutoTokenizer.from_pretrained(MODEL_ID, use_fast=True)
mdl = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb,
    device_map="auto",
)

# beberapa model Qwen perlu pad_token; set ke eos untuk aman
if tok.pad_token_id is None and tok.eos_token_id is not None:
    tok.pad_token_id = tok.eos_token_id

EOS_ID = tok.convert_tokens_to_ids("<|im_end|>") or tok.eos_token_id

SYSTEM_PROMPT = (
    "You are a helpful assistant that writes concise two-paragraph summaries in Indonesian. "
    "Prioritize key topics, arguments, and decisions; avoid meta commentary."
)

# ============================
# 3) HELPER FUNCTIONS
# ============================

def _apply_chat_and_generate(messages, max_new_tokens=320, max_ctx=MAX_CTX):
    """
    messages: list of dicts [{"role":"system"/"user"/"assistant", "content":"..."}]
    return: ONLY the newly generated assistant text (tanpa echo prompt)
    """
    prompt_text = tok.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = tok(prompt_text, return_tensors="pt", truncation=True, max_length=max_ctx).to(mdl.device)

    out = mdl.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        eos_token_id=EOS_ID,
        pad_token_id=tok.pad_token_id,
        **GEN_KW
    )
    # ambil hanya token yang di-generate (slice setelah panjang input)
    gen_tokens = out[0][inputs["input_ids"].shape[-1]:]
    text = tok.decode(gen_tokens, skip_special_tokens=True).strip()
    return text

def _clean(text: str) -> str:
    # Bersihkan artefak umum
    t = text.strip()
    t = re.sub(r"^(Hasil akhir:?\s*)", "", t, flags=re.IGNORECASE)
    t = re.sub(r"\s+\n", "\n", t)
    t = re.sub(r"\n{3,}", "\n\n", t)
    return t.strip()

def tokenize_len(text: str) -> int:
    return len(tok.encode(text, add_special_tokens=False))

def chunk_by_tokens(text: str, max_tokens=CHUNK_TOKENS, overlap=OVERLAP_TOKENS):
    ids = tok.encode(text, add_special_tokens=False)
    chunks = []
    i, n = 0, len(ids)
    while i < n:
        j = min(i + max_tokens, n)
        chunks.append(tok.decode(ids[i:j], skip_special_tokens=True))
        if j >= n:
            break
        i = j - overlap
    return chunks

def map_summarize(chunk: str, max_new=MAP_MAX_NEW) -> str:
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content":
         "Ringkas percakapan berikut menjadi 2–4 kalimat yang fokus pada inti topik, argumen, "
         "fakta kunci, dan keputusan (jika ada). Tulis dalam Bahasa Indonesia yang natural.\n\n"
         + chunk}
    ]
    return _clean(_apply_chat_and_generate(messages, max_new_tokens=max_new))

def reduce_summaries(maps: list[str], max_new=REDUCE_MAX_NEW) -> str:
    joined = "\n".join(maps)
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content":
         "Gabungkan ringkasan per-bagian berikut menjadi DUA paragraf (total ~75–150 kata), jelas, "
         "padat, dan mewakili topik utama + poin pendukung. Hindari kalimat meta dan hindari pengulangan.\n\n"
         + joined + "\n\nHasil akhir:"}
    ]
    return _clean(_apply_chat_and_generate(messages, max_new_tokens=max_new))

def summarize_text_to_two_paragraphs(text: str,
                                     chunk_tokens=CHUNK_TOKENS,
                                     overlap_tokens=OVERLAP_TOKENS) -> str:
    txt = (text or "").strip()
    if not txt:
        return ""

    try:
        if tokenize_len(txt) < chunk_tokens:
            m = map_summarize(txt)
            return reduce_summaries([m])
        chunks = chunk_by_tokens(txt, chunk_tokens, overlap_tokens)
        maps = [map_summarize(c) for c in chunks]
        return reduce_summaries(maps)
    except torch.cuda.OutOfMemoryError:
        # fallback: coba dengan chunk lebih kecil
        torch.cuda.empty_cache()
        if chunk_tokens > FALLBACK_CHUNK:
            return summarize_text_to_two_paragraphs(txt, chunk_tokens=FALLBACK_CHUNK, overlap_tokens=overlap_tokens)
        raise

def safe_save_csv(df: pd.DataFrame, path: str):
    # simpan ke file temp lalu rename (mengurangi risiko korup)
    tmp = path + ".tmp"
    df.to_csv(tmp, index=False)
    os.replace(tmp, path)

# ============================
# 4) LOAD / RESUME DATA
# ============================
assert os.path.exists(INPUT_CSV), f"Tidak menemukan file: {INPUT_CSV}"

if os.path.exists(OUTPUT_CSV):
    # lanjutkan dari file hasil sebelumnya
    out_df = pd.read_csv(OUTPUT_CSV)
    in_df  = pd.read_csv(INPUT_CSV)

    # pastikan sama panjang/urut; merge by index jika struktur sama
    if len(out_df) == len(in_df) and all(col in out_df.columns for col in in_df.columns):
        df = out_df
        # kalau kolom summary belum ada, buat
        if SUMMARY_COL not in df.columns:
            df[SUMMARY_COL] = ""
    else:
        # fallback: join ke input by key 'id' kalau ada
        df = in_df.copy()
        if SUMMARY_COL in out_df.columns and "id" in out_df.columns and "id" in df.columns:
            df = df.merge(out_df[["id", SUMMARY_COL]], on="id", how="left")
        if SUMMARY_COL not in df.columns:
            df[SUMMARY_COL] = ""
else:
    df = pd.read_csv(INPUT_CSV)
    if SUMMARY_COL not in df.columns:
        df[SUMMARY_COL] = ""

# sanity check
assert TEXT_COL in df.columns, f"Kolom '{TEXT_COL}' tidak ada di CSV!"

# ============================
# 5) PROCESS & INCREMENTAL SAVE
# ============================
total = len(df)
print(f"Mulai summarization: {total} baris. Output → {OUTPUT_CSV}")

progress_each = 1   # simpan setiap selesai 1 baris
counter_since_save = 0

for idx in tqdm(range(total), desc="Summarizing (incremental save)"):
    # skip jika sudah ada ringkasan non-kosong
    if isinstance(df.at[idx, SUMMARY_COL], str) and df.at[idx, SUMMARY_COL].strip():
        continue

    text_val = df.at[idx, TEXT_COL]
    try:
        summary = summarize_text_to_two_paragraphs(text_val)
    except torch.cuda.OutOfMemoryError:
        torch.cuda.empty_cache()
        try:
            summary = summarize_text_to_two_paragraphs(text_val, chunk_tokens=FALLBACK_CHUNK)
        except Exception as e:
            summary = f"[ERROR: OOM Fallback] {e}"
    except Exception as e:
        summary = f"[ERROR] {e}"

    df.at[idx, SUMMARY_COL] = summary
    counter_since_save += 1

    # incremental save setiap N baris (di sini = 1 baris)
    if counter_since_save >= progress_each:
        safe_save_csv(df, OUTPUT_CSV)
        counter_since_save = 0

# save terakhir
safe_save_csv(df, OUTPUT_CSV)
print("Selesai. Disimpan ke:", OUTPUT_CSV)

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

Mulai summarization: 998 baris. Output → data_with_summary.csv


Summarizing (incremental save):   0%|          | 0/998 [00:00<?, ?it/s]

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature', 'top_p'

KeyboardInterrupt: 

In [ ]:
df_out = pd.read_csv(OUTPUT_CSV)
df_out['summary_2para'].iloc[0]

'Rencana latihan ini dirancang khusus untuk lansia yang tinggal di rumah, mencakup lima aktivitas utama yang bertujuan memperkuat dan meningkatkan fleksibilitas tubuh. Aktivitas pertama adalah gerakan seat to stand untuk membantu lansia berdiri dari duduk dengan aman, disusul oleh trunk extension untuk menguatkan otot perut sambil tetap duduk. Tekuk punggung ke depan dan kemudian tegakkan dilakukan untuk meningkatkan fleksibilitas, sementara incline push up menggunakan tembok melatih dada dan lengan, dan seated overhead press dengan botol atau dumbbell melatih otot punggung dan lengan. Rencana latihan ini direkomendasikan untuk dilakukan dua hingga tiga kali seminggu untuk hasil optimal.\nHasil akhir dari pelaksanaan rencana latihan ini adalah peningkatan kekuatan otot, fleksibilitas, dan keseimbangan pada lansia, yang dapat membantu mereka menjalani hidup sehari-hari dengan lebih mandiri dan sehat. Selain itu, rutinitas latihan ini juga dapat mencegah penurunan fungsi fisik dan mengur

# Modeling

In [ ]:
!pip install av

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 39.9/39.9 MB 12.4 MB/s eta 0:00:00


## Baseline Video


### Configuration

In [ ]:
import os
import torch
import av
import numpy as np
from torch.utils.data import Dataset, DataLoader
from transformers import AutoImageProcessor

# Konfigurasi
# -----------------------------
DATA_DIR = "./data/videos_train"
MODEL_NAME = "facebook/timesformer-base-finetuned-k400"
MAX_FRAMES = 8       # Jumlah frame yang akan di-sample dari video
BATCH_SIZE = 4
IMG_SIZE = 224

# Fungsi bantu
# -----------------------------
def read_video_pyav(container, indices):
    frames = []
    container.seek(0)
    # Efisiensi: Hanya decode frame yang dibutuhkan
    for i, frame in enumerate(container.decode(video=0)):
        if i > indices[-1]:
            break
        if i in indices:
            frames.append(frame.to_ndarray(format="rgb24"))
    return np.stack(frames)

def sample_frame_indices(clip_len, total_frames):
    # Memastikan kita punya cukup frame
    if total_frames < clip_len:
        # Jika video terlalu pendek, kita ulangi frame yang ada
        indices = np.linspace(0, total_frames - 1, num=clip_len, dtype=int)
    else:
        # Jika video cukup panjang, ambil segmen acak
        start_idx = np.random.randint(0, total_frames - clip_len + 1)
        indices = np.arange(start_idx, start_idx + clip_len)
    return indices

In [ ]:
# Custom Dataset
# -----------------------------
class EmotionVideoDataset(Dataset):
    def __init__(self, dataframe, image_processor, clip_len=MAX_FRAMES):
        # --- PERUBAHAN DI SINI ---
        # Kita sekarang menerima DataFrame, bukan data_dir
        self.dataframe = dataframe
        self.image_processor = image_processor
        self.clip_len = clip_len

        # Ambil path video dan label langsung dari kolom DataFrame
        self.video_paths = self.dataframe['video_path'].tolist()
        self.labels = self.dataframe['emotion_label'].tolist()

        # Mapping label tetap berguna untuk konsistensi
        self.label2id = {
            'Proud': 0, 'Trust': 1, 'Joy': 2, 'Surprise': 3,
            'Neutral': 4, 'Sadness': 5, 'Fear': 6, 'Anger': 7
        }
        self.id2label = {v: k for k, v in self.label2id.items()}

        print(f"✅ Dataset diinisialisasi dengan {len(self.video_paths)} video dari DataFrame.")

    def __len__(self):
        return len(self.video_paths)

    def __getitem__(self, idx):
        video_path = self.video_paths[idx]
        label = self.labels[idx]

        try:
            container = av.open(video_path)
            total_frames = container.streams.video[0].frames
            # Jika metadata frame tidak ada, estimasi dari durasi & fps
            if total_frames <= 0:
                total_frames = int(container.streams.video[0].average_rate * container.duration / 1_000_000)

            indices = sample_frame_indices(self.clip_len, total_frames)
            video_frames = read_video_pyav(container, indices)
            container.close()

            # ✅ Preprocess dengan ImageProcessor
            # Inputnya adalah list dari numpy array [H, W, C]
            # Outputnya adalah dict dengan 'pixel_values'
            inputs = self.image_processor(list(video_frames), return_tensors="pt")

            # ✅ Dapatkan tensor pixel_values. Shape-nya akan menjadi [T, C, H, W]
            pixel_values = inputs["pixel_values"].squeeze(0)

            return {
                "pixel_values": pixel_values,
                "labels": torch.tensor(label, dtype=torch.long)
            }

        except Exception as e:
            print(f"Error reading {video_path}: {e}. Returning dummy data.")
            # ✅ Pastikan dummy data punya shape yang benar [T, C, H, W]
            dummy_video = torch.zeros((self.clip_len, 3, IMG_SIZE, IMG_SIZE))
            return {
                "pixel_values": dummy_video,
                "labels": torch.tensor(label, dtype=torch.long)
            }

# Collate Function (Sangat Penting!)
# -----------------------------
def collate_fn(batch):
    # Stack semua tensor dari dataset
    # Shape yang dihasilkan kemungkinan besar sudah benar: [B, C, T, H, W]
    pixel_values = torch.stack([item["pixel_values"] for item in batch])

    labels = torch.stack([item["labels"] for item in batch])

    return {"pixel_values": pixel_values, "labels": labels}

In [ ]:
# !pip install -q decord
import os, shutil, hashlib
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
from decord import VideoReader, cpu

# ---- util sampling ----
def select_indices(total_frames: int, clip_len: int, strategy: str = "uniform", window_ratio: float = 0.2):
    """
    Pilih index frame yang akan diambil.
    - uniform: sebar merata dari awal–akhir video
    - rand_window: ambil window acak (proporsi window_ratio dari total), lalu sampling uniform di dalam window tsb.
    """
    total_frames = int(max(1, total_frames))
    clip_len = int(max(1, clip_len))

    if total_frames <= clip_len:
        return np.linspace(0, total_frames - 1, num=clip_len, dtype=int)

    if strategy == "rand_window":
        win = max(clip_len, int(total_frames * window_ratio))
        start = np.random.randint(0, max(1, total_frames - win + 1))
        end = start + win - 1
        return np.linspace(start, end, num=clip_len, dtype=int)

    # default uniform full-length
    return np.linspace(0, total_frames - 1, num=clip_len, dtype=int)

# ---- util cache lokal (opsional, sangat direkomendasikan) ----
def cache_to_local(path: str, cache_dir: str = "/content/.vcache/videos") -> str:
    """
    Copy-once: pertama kali akses file dari Drive → salin ke SSD lokal Colab.
    Akses berikutnya langsung dari SSD (kenceng). Tetap "pakai Drive" sbg sumber.
    """
    if not path or not os.path.isfile(path):
        return path  # biarin error handling di __getitem__

    os.makedirs(cache_dir, exist_ok=True)
    # gunakan hash path agar unik & tahan collision
    h = hashlib.md5(path.encode()).hexdigest()[:10]
    local_name = f"{h}__{os.path.basename(path)}"
    local_path = os.path.join(cache_dir, local_name)

    if not os.path.exists(local_path) or os.path.getsize(local_path) == 0:
        # copy pertama kali
        shutil.copy2(path, local_path)
    return local_path

# -----------------------------
# Custom Dataset (versi cepat)
# -----------------------------
class EmotionVideoDataset(Dataset):
    def __init__(
        self,
        dataframe,
        image_processor,
        clip_len=8,
        backend="decord",                 # 'decord' (disarankan) / 'pyav'
        sampling="rand_window",           # 'uniform' / 'rand_window'
        window_ratio=0.2,                 # hanya dipakai kalau sampling='rand_window'
        use_local_cache=True,             # cache sekali ke SSD lokal
        cache_dir="/content/.vcache/videos",
        img_size=224
    ):
        self.df = dataframe.reset_index(drop=True)
        self.image_processor = image_processor
        self.clip_len = clip_len
        self.backend = backend
        self.sampling = sampling
        self.window_ratio = window_ratio
        self.use_local_cache = use_local_cache
        self.cache_dir = cache_dir
        self.img_size = img_size

        self.video_paths = self.df["video_path"].tolist()
        self.labels = self.df["emotion_label"].tolist()

        # mapping label (optional, buat referensi)
        self.label2id = {
            'Proud': 0, 'Trust': 1, 'Joy': 2, 'Surprise': 3,
            'Neutral': 4, 'Sadness': 5, 'Fear': 6, 'Anger': 7
        }
        self.id2label = {v: k for k, v in self.label2id.items()}

        print(f"✅ Dataset diinisialisasi: {len(self.video_paths)} video | backend={backend} | sampling={sampling} | cache={use_local_cache}")

    def __len__(self):
        return len(self.video_paths)

    def _load_frames_decord(self, path):
        vr = VideoReader(path, ctx=cpu(0))
        total = len(vr)
        idxs = select_indices(total, self.clip_len, strategy=self.sampling, window_ratio=self.window_ratio)
        frames = vr.get_batch(idxs).asnumpy()  # (T, H, W, C) RGB uint8
        return frames

    def _load_frames_pyav(self, path):
        import av  # lazy import
        container = av.open(path)
        total_frames = container.streams.video[0].frames
        if not total_frames or total_frames <= 0:
            # fallback estimasi kasar
            total_frames = int(container.streams.video[0].average_rate * container.duration / 1_000_000)
        idxs = select_indices(total_frames, self.clip_len, strategy=self.sampling, window_ratio=self.window_ratio)

        frames = []
        container.seek(0)
        # decode minimum yang diperlukan (tetap bisa mahal di pyav)
        need = set(int(i) for i in idxs)
        for i, frame in enumerate(container.decode(video=0)):
            if i > idxs[-1]:
                break
            if i in need:
                frames.append(frame.to_ndarray(format="rgb24"))
        container.close()

        if len(frames) < self.clip_len:
            # pad repeat kalau kurang
            if frames:
                last = frames[-1]
                while len(frames) < self.clip_len:
                    frames.append(last)
            else:
                frames = [np.zeros((self.img_size, self.img_size, 3), dtype=np.uint8) for _ in range(self.clip_len)]
        return np.stack(frames, axis=0)  # (T, H, W, C)

    def __getitem__(self, idx):
        video_path = self.video_paths[idx]
        label = int(self.labels[idx])

        try:
            # tetap sumber di Drive, tapi cache lokal sekali biar kenceng
            local_path = cache_to_local(video_path, self.cache_dir) if self.use_local_cache else video_path

            if self.backend == "decord":
                frames = self._load_frames_decord(local_path)
            else:
                frames = self._load_frames_pyav(local_path)

            # preprocess → (T, C, H, W); processor Timesformer akan handle resize/normalize
            inputs = self.image_processor(list(frames), return_tensors="pt")
            pixel_values = inputs["pixel_values"].squeeze(0)  # (T, C, H, W)

        except Exception as e:
            print(f"[WARN] Error reading {video_path}: {e}. Using dummy.")
            pixel_values = torch.zeros((self.clip_len, 3, self.img_size, self.img_size), dtype=torch.float32)

        return {
            "pixel_values": pixel_values,                     # (T, C, H, W)
            "labels": torch.tensor(label, dtype=torch.long)
        }

# Collate Function (penting!)
def collate_fn(batch):
    """
    Timesformer (HF) mengharapkan shape (B, T, C, H, W).
    Kita stack list [(T,C,H,W), ...] → (B,T,C,H,W)
    """
    pixel_values = torch.stack([b["pixel_values"] for b in batch], dim=0)  # (B, T, C, H, W)
    labels = torch.stack([b["labels"] for b in batch])
    return {"pixel_values": pixel_values, "labels": labels}


### Data Preparation

In [ ]:
from sklearn.model_selection import train_test_split
from transformers import AutoImageProcessor
from torch.utils.data import DataLoader

df_all = pd.read_csv("./data/df_train_processed.csv")

# gunakan video yang memiliki video_path
df_all = df_all.dropna(subset=['video_path'])

# sesuaikan dengan path pada directory colab
df_all['video_path'] = df_all['video_path'].str.replace(
    '../data/raw/videos_train/',
    './data/videos_train/',
    regex=False
)

# 1. Lakukan Stratified Split (seperti yang dibahas sebelumnya)
df_train, df_temp = train_test_split(
    df_all,
    test_size=0.3, # 30% untuk validation + test
    random_state=42,
    stratify=df_all['emotion_label']
)

df_val, df_test = train_test_split(
    df_temp,
    test_size=0.3, # 30% dari 30% -> 20% untuk val, 10% untuk test
    random_state=42,
    stratify=df_temp['emotion_label']
)

# 2. Inisialisasi Image Processor
image_processor = AutoImageProcessor.from_pretrained(MODEL_NAME)

# 3. Buat objek Dataset untuk setiap set menggunakan DataFrame
print("Membuat dataset training...")
train_dataset = EmotionVideoDataset(dataframe=df_train, image_processor=image_processor)

print("\nMembuat dataset validasi...")
val_dataset = EmotionVideoDataset(dataframe=df_val, image_processor=image_processor)

print("\nMembuat dataset testing...")
test_dataset = EmotionVideoDataset(dataframe=df_test, image_processor=image_processor)

# 4. Buat DataLoader seperti biasa
train_dataloader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_fn)
val_dataloader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn)
test_dataloader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn)

print(f"\n✅ Siap untuk training! Ukuran DataLoader: Train={len(train_dataloader)}, Val={len(val_dataloader)}")

Fetching 1 files:   0%|          | 0/1 [00:00<?, ?it/s]

Membuat dataset training...
✅ Dataset diinisialisasi: 546 video | backend=decord | sampling=rand_window | cache=True

Membuat dataset validasi...
✅ Dataset diinisialisasi: 163 video | backend=decord | sampling=rand_window | cache=True

Membuat dataset testing...
✅ Dataset diinisialisasi: 71 video | backend=decord | sampling=rand_window | cache=True

✅ Siap untuk training! Ukuran DataLoader: Train=137, Val=41


### Training

In [ ]:
from sklearn.metrics import f1_score, accuracy_score, precision_score, recall_score

def custom_metric(eval_pred):
    labels = eval_pred.label_ids
    preds = eval_pred.predictions.argmax(-1)
    return {
        "macro_f1": f1_score(labels, preds, average="macro"),
        "accuracy": accuracy_score(labels, preds),
        "precision_macro": precision_score(labels, preds, average="macro", zero_division=0),
        "recall_macro": recall_score(labels, preds, average="macro", zero_division=0),
    }


In [ ]:
import glob, os
from transformers import Trainer, TrainingArguments

# 1) Cari checkpoint terakhir (karena save_strategy='epoch', harusnya ada tiap akhir epoch)
ckpts = sorted(glob.glob("./training/timesformer_2/checkpoint-*"), key=os.path.getmtime)
last_ckpt = ckpts[-1] if ckpts else None
print("Last ckpt:", last_ckpt)

Last ckpt: None


In [ ]:
image_processor = AutoImageProcessor.from_pretrained("facebook/timesformer-base-finetuned-k400")
model = TimesformerForVideoClassification.from_pretrained("facebook/timesformer-base-finetuned-k400")

# 2) Update argumen jadi 5 epoch, tetap pakai output_dir yang sama
new_args = TrainingArguments(
    output_dir="./training/timesformer",
    overwrite_output_dir=False,
    push_to_hub=False,
    report_to='none',

    learning_rate=5e-6,
    num_train_epochs=5,                    # <── dari 2 -> 5
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    dataloader_num_workers=8,

    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="epoch",

    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    greater_is_better=True,

    # opsional biar ga numpuk banyak checkpoint
    # save_total_limit=2,
)

# 3) Recreate Trainer dengan args baru (model/dataset/metric tetap sama)
trainer = Trainer(
    model=model,
    args=new_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=custom_metric,
)

# 4) Lanjutkan training dari checkpoint terakhir (auto load optimizer & scheduler state)
if last_ckpt:
    trainer.train(resume_from_checkpoint=last_ckpt)
else:
    # kalau (aneh tapi mungkin) belum ada checkpoint, ya jalan tetap — tapi optimizer state mulai baru
    trainer.train()

Fetching 1 files:   0%|          | 0/1 [00:00<?, ?it/s]

config.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/486M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/486M [00:00<?, ?B/s]

[WARN] Error reading ./data/videos_train/Trust/330.mp4: [23:28:32] /github/workspace/src/video/ffmpeg/threaded_decoder.cc:292: [23:28:32] /github/workspace/src/video/ffmpeg/threaded_decoder.cc:218: Check failed: avcodec_send_packet(dec_ctx_.get(), pkt.get()) >= 0 (-11 vs. 0) Thread worker: Error sending packet.. Using dummy.
[WARN] Error reading ./data/videos_train/Surprise/4.mp4: [23:28:45] /github/workspace/src/video/ffmpeg/threaded_decoder.cc:292: [23:28:45] /github/workspace/src/video/ffmpeg/threaded_decoder.cc:218: Check failed: avcodec_send_packet(dec_ctx_.get(), pkt.get()) >= 0 (-11 vs. 0) Thread worker: Error sending packet.. Using dummy.


Epoch,Training Loss,Validation Loss,Macro F1,Accuracy,Precision Macro,Recall Macro
1,4.265700,2.159260,0.085761,0.380368,0.092072,0.110481
2,1.874800,1.867226,0.130155,0.386503,0.122825,0.144380
3,1.529600,1.805382,0.122874,0.343558,0.116226,0.136000


[WARN] Error reading ./data/videos_train/Trust/348.mp4: [23:30:06] /github/workspace/src/video/ffmpeg/threaded_decoder.cc:292: [23:30:06] /github/workspace/src/video/ffmpeg/threaded_decoder.cc:218: Check failed: avcodec_send_packet(dec_ctx_.get(), pkt.get()) >= 0 (-11 vs. 0) Thread worker: Error sending packet.. Using dummy.
[WARN] Error reading ./data/videos_train/Surprise/328.mp4: [23:30:21] /github/workspace/src/video/ffmpeg/threaded_decoder.cc:292: [23:30:21] /github/workspace/src/video/ffmpeg/threaded_decoder.cc:218: Check failed: avcodec_send_packet(dec_ctx_.get(), pkt.get()) >= 0 (-11 vs. 0) Thread worker: Error sending packet.. Using dummy.
[WARN] Error reading ./data/videos_train/Surprise/350.mp4: [23:33:40] /github/workspace/src/video/ffmpeg/threaded_decoder.cc:292: [23:33:40] /github/workspace/src/video/ffmpeg/threaded_decoder.cc:218: Check failed: avcodec_send_packet(dec_ctx_.get(), pkt.get()) >= 0 (-11 vs. 0) Thread worker: Error sending packet.. Using dummy.
[WARN] Error 

KeyboardInterrupt: 

In [ ]:
# Simpan model terbaik + config
trainer.save_model("./training/timesformer_2/best")
# Simpan processor juga biar inference gampang
image_processor.save_pretrained("./training/timesformer_2/best")
# (opsional) simpan label mapping kalau kamu set di config
# model.config.label2id, model.config.id2label sudah ikut tersimpan

['./training/timesformer_2/best/preprocessor_config.json']

In [ ]:
from transformers import TimesformerForVideoClassification, AutoImageProcessor
proc  = AutoImageProcessor.from_pretrained("./training/timesformer/best")
model = TimesformerForVideoClassification.from_pretrained("./training/timesformer/best")

## Baseline Audio

In [ ]:
# === INSTALL (jika perlu) ===
!pip install -q transformers torchaudio accelerate datasets

In [ ]:
import os
import torch
import torchaudio
import numpy as np
import pandas as pd
from dataclasses import dataclass
from typing import Any, Dict, List

from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, accuracy_score, precision_score, recall_score

from transformers import (
    AutoFeatureExtractor,
    AutoModelForAudioClassification,
    TrainingArguments,
    Trainer,
)

# -----------------------------
# KONFIG
# -----------------------------
NUM_CLASSES = 8
TARGET_SR = 16000
MAX_SECONDS = 10.0                     # potong/padang ke ~6 detik
MAX_LEN = int(TARGET_SR * MAX_SECONDS)
MODEL_NAME_AUDIO = "MIT/ast-finetuned-audioset-10-10-0.4593"  # AST
SEED = 42

label2id = {
    'Proud': 0, 'Trust': 1, 'Joy': 2, 'Surprise': 3,
    'Neutral': 4, 'Sadness': 5, 'Fear': 6, 'Anger': 7
}
id2label = {v: k for k, v in label2id.items()}

# -----------------------------
# LOAD CSV & SPLIT
# -----------------------------
df_all = pd.read_csv("./data/df_train_processed.csv")

# Keep baris yang punya audio_path
df_all = df_all.dropna(subset=["audio_path"]).copy()

# Split stratified by label (konsisten dengan video)
df_train, df_temp = train_test_split(
    df_all, test_size=0.3, random_state=SEED, stratify=df_all["emotion_label"]
)
df_val, df_test = train_test_split(
    df_temp, test_size=0.3, random_state=SEED, stratify=df_temp["emotion_label"]
)

print("Train/Val/Test sizes (audio):", len(df_train), len(df_val), len(df_test))

Train/Val/Test sizes (audio): 546 163 71


In [ ]:
# -----------------------------
# DATASET
# -----------------------------
class EmotionAudioDataset(torch.utils.data.Dataset):
    def __init__(self, df, feature_extractor, target_sr=TARGET_SR, max_len=MAX_LEN):
        self.df = df.reset_index(drop=True)
        self.fe = feature_extractor
        self.target_sr = feature_extractor.sampling_rate
        self.max_len = max_len

    def _load_wav(self, path):
        wav, sr = torchaudio.load(path)  # (C, N)
        if wav.shape[0] > 1:
            wav = wav.mean(dim=0, keepdim=True)     # mono
        if sr != self.target_sr:
            wav = torchaudio.functional.resample(wav, sr, self.target_sr)
        wav = wav.squeeze(0)  # (N,)

        # center-crop / pad to fixed length
        n = wav.shape[0]
        if n > self.max_len:
            start = (n - self.max_len) // 2
            wav = wav[start:start + self.max_len]
        elif n < self.max_len:
            wav = torch.nn.functional.pad(wav, (0, self.max_len - n))
        return wav

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        y = int(row["emotion_label"])
        path = row["audio_path"]

        try:
            wav = self._load_wav(path)
        except Exception as e:
            # Fallback jika file rusak
            print(f"[WARN] Error load {path}: {e}. Using silence.")
            wav = torch.zeros(self.max_len, dtype=torch.float32)

        # AST FeatureExtractor expects raw waveform float + sampling_rate
        inputs = self.fe(wav.numpy(), sampling_rate=self.fe.sampling_rate, return_tensors="pt")
        item = {k: v.squeeze(0) for k, v in inputs.items()}
        item["labels"] = torch.tensor(y, dtype=torch.long)
        return item

    def __len__(self):
        return len(self.df)

# -----------------------------
# COLLATOR (pad by extractor)
# -----------------------------
@dataclass
class DataCollatorAudio:
    feature_extractor: Any
    def __call__(self, features: List[Dict[str, Any]]) -> Dict[str, torch.Tensor]:
        labels = torch.tensor([f.pop("labels") for f in features], dtype=torch.long)
        batch = self.feature_extractor.pad(features, return_tensors="pt")
        batch["labels"] = labels
        return batch

### MIT/ast-finetuned-audioset-10-10-0.4593

In [ ]:
MODEL_NAME_AUDIO = "MIT/ast-finetuned-audioset-10-10-0.4593"  # AST

# -----------------------------
# DATALOADERS
# -----------------------------
fe = AutoFeatureExtractor.from_pretrained(MODEL_NAME_AUDIO)
# Pastikan sampling_rate feature extractor = 16000
if fe.sampling_rate != TARGET_SR:
    fe.sampling_rate = TARGET_SR  # supaya konsisten

train_ds = EmotionAudioDataset(df_train, fe)
val_ds   = EmotionAudioDataset(df_val, fe)
test_ds  = EmotionAudioDataset(df_test, fe)

collator = DataCollatorAudio(feature_extractor=fe)

# -----------------------------
# MODEL (head baru 8 kelas)
# -----------------------------
import torch.nn as nn

model_audio = AutoModelForAudioClassification.from_pretrained(
    MODEL_NAME_AUDIO,
    num_labels=NUM_CLASSES,
    label2id=label2id,
    id2label=id2label,
    problem_type="single_label_classification",
    ignore_mismatched_sizes=True,  # penting: ganti head
)

# -----------------------------
# CLASS WEIGHTS (imbalanced)
# -----------------------------
counts = np.bincount(df_train["emotion_label"].values, minlength=NUM_CLASSES)
# bobot ~ total/(K * count)
class_weights = (counts.sum() / (counts + 1e-6)) / NUM_CLASSES
class_weights = torch.tensor(class_weights, dtype=torch.float)

# -----------------------------
# METRICS
# -----------------------------
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = logits.argmax(-1)
    return {
        "macro_f1": f1_score(labels, preds, average="macro"),
        "accuracy": accuracy_score(labels, preds),
        "precision_macro": precision_score(labels, preds, average="macro", zero_division=0),
        "recall_macro": recall_score(labels, preds, average="macro", zero_division=0),
    }

# -----------------------------
# TRAINER dengan weighted CE
# -----------------------------
from transformers import Trainer

import torch.nn as nn
from transformers import Trainer

class WeightedTrainer(Trainer):
    def __init__(self, class_weights, **kwargs):
        super().__init__(**kwargs)
        self.class_weights = class_weights

    # Tambahkan num_items_in_batch dengan default None (atau **kwargs)
    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        # Jangan mutasi inputs dari luar secara permanen (copy ringan)
        labels = inputs.get("labels")
        # Buang labels saat forward
        _inputs = {k: v for k, v in inputs.items() if k != "labels"}

        outputs = model(**_inputs)
        logits = outputs.logits

        loss_fct = nn.CrossEntropyLoss(weight=self.class_weights.to(logits.device))
        # pastikan shape sesuai (N, num_labels)
        loss = loss_fct(logits.view(-1, self.model.config.num_labels),
                        labels.view(-1))

        return (loss, outputs) if return_outputs else loss

args_audio = TrainingArguments(
    output_dir="./training/ast_audio",
    overwrite_output_dir=False,
    learning_rate=3e-5,
    num_train_epochs=5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    dataloader_num_workers=4,
    fp16=True,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    greater_is_better=True,
    save_total_limit=2,
    report_to="none",
    seed=SEED,
)

trainer_audio = WeightedTrainer(
    class_weights=class_weights,
    model=model_audio,
    args=args_audio,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    data_collator=collator,
    compute_metrics=compute_metrics,
)

trainer_audio.train()
# Optional: simpan best
trainer_audio.save_model("./training/ast_audio_best")


Train/Val/Test sizes (audio): 546 163 71


Fetching 1 files:   0%|          | 0/1 [00:00<?, ?it/s]

preprocessor_config.json:   0%|          | 0.00/297 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/346M [00:00<?, ?B/s]

Some weights of ASTForAudioClassification were not initialized from the model checkpoint at MIT/ast-finetuned-audioset-10-10-0.4593 and are newly initialized because the shapes did not match:
- classifier.dense.bias: found shape torch.Size([527]) in the checkpoint and torch.Size([8]) in the model instantiated
- classifier.dense.weight: found shape torch.Size([527, 768]) in the checkpoint and torch.Size([8, 768]) in the model instantiated
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Epoch,Training Loss,Validation Loss,Macro F1,Accuracy,Precision Macro,Recall Macro
1,2.354300,2.047690,0.088001,0.208589,0.324131,0.123101
2,1.657700,2.106179,0.126130,0.196319,0.136236,0.193864
3,0.947600,2.126105,0.169497,0.269939,0.176366,0.180178
4,0.416000,2.334604,0.148220,0.306748,0.163585,0.145772
5,0.189500,2.594193,0.155255,0.343558,0.161833,0.157951


### MIT/ast-finetuned-audioset-14-14-0.443

In [ ]:
MODEL_NAME_AUDIO = "MIT/ast-finetuned-audioset-14-14-0.443"  # AST

# -----------------------------
# DATALOADERS
# -----------------------------
fe = AutoFeatureExtractor.from_pretrained(MODEL_NAME_AUDIO)
# Pastikan sampling_rate feature extractor = 16000
# if fe.sampling_rate != TARGET_SR:
#     fe.sampling_rate = TARGET_SR  # supaya konsisten

train_ds = EmotionAudioDataset(df_train, fe)
val_ds   = EmotionAudioDataset(df_val, fe)
test_ds  = EmotionAudioDataset(df_test, fe)

collator = DataCollatorAudio(feature_extractor=fe)

# -----------------------------
# MODEL (head baru 8 kelas)
# -----------------------------
import torch.nn as nn

model_audio = AutoModelForAudioClassification.from_pretrained(
    MODEL_NAME_AUDIO,
    num_labels=NUM_CLASSES,
    label2id=label2id,
    id2label=id2label,
    problem_type="single_label_classification",
    ignore_mismatched_sizes=True,  # penting: ganti head
)

# -----------------------------
# CLASS WEIGHTS (imbalanced)
# -----------------------------
counts = np.bincount(df_train["emotion_label"].values, minlength=NUM_CLASSES)
# bobot ~ total/(K * count)
class_weights = (counts.sum() / (counts + 1e-6)) / NUM_CLASSES
class_weights = torch.tensor(class_weights, dtype=torch.float)

# -----------------------------
# METRICS
# -----------------------------
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = logits.argmax(-1)
    return {
        "macro_f1": f1_score(labels, preds, average="macro"),
        "accuracy": accuracy_score(labels, preds),
        "precision_macro": precision_score(labels, preds, average="macro", zero_division=0),
        "recall_macro": recall_score(labels, preds, average="macro", zero_division=0),
    }

# -----------------------------
# TRAINER dengan weighted CE
# -----------------------------
from transformers import Trainer

import torch.nn as nn
from transformers import Trainer

class WeightedTrainer(Trainer):
    def __init__(self, class_weights, **kwargs):
        super().__init__(**kwargs)
        self.class_weights = class_weights

    # Tambahkan num_items_in_batch dengan default None (atau **kwargs)
    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        # Jangan mutasi inputs dari luar secara permanen (copy ringan)
        labels = inputs.get("labels")
        # Buang labels saat forward
        _inputs = {k: v for k, v in inputs.items() if k != "labels"}

        outputs = model(**_inputs)
        logits = outputs.logits

        loss_fct = nn.CrossEntropyLoss(weight=self.class_weights.to(logits.device))
        # pastikan shape sesuai (N, num_labels)
        loss = loss_fct(logits.view(-1, self.model.config.num_labels),
                        labels.view(-1))

        return (loss, outputs) if return_outputs else loss

args_audio = TrainingArguments(
    output_dir="./training/ast_audio_443",
    overwrite_output_dir=False,
    learning_rate=3e-5,
    num_train_epochs=5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    dataloader_num_workers=4,
    fp16=True,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    greater_is_better=True,
    save_total_limit=2,
    report_to="none",
    seed=SEED,
)

trainer_audio = WeightedTrainer(
    class_weights=class_weights,
    model=model_audio,
    args=args_audio,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    data_collator=collator,
    compute_metrics=compute_metrics,
)

trainer_audio.train()
# Optional: simpan best
trainer_audio.save_model("./training/ast433_audio_best")


Fetching 1 files:   0%|          | 0/1 [00:00<?, ?it/s]

preprocessor_config.json:   0%|          | 0.00/297 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/345M [00:00<?, ?B/s]

Some weights of ASTForAudioClassification were not initialized from the model checkpoint at MIT/ast-finetuned-audioset-14-14-0.443 and are newly initialized because the shapes did not match:
- classifier.dense.bias: found shape torch.Size([527]) in the checkpoint and torch.Size([8]) in the model instantiated
- classifier.dense.weight: found shape torch.Size([527, 768]) in the checkpoint and torch.Size([8, 768]) in the model instantiated
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Epoch,Training Loss,Validation Loss,Macro F1,Accuracy,Precision Macro,Recall Macro
1,2.355600,2.016849,0.084929,0.361963,0.077238,0.116125
2,1.904600,2.115700,0.089994,0.141104,0.087329,0.186571
3,1.373700,2.275168,0.203757,0.294479,0.294573,0.195382
4,0.693900,2.497885,0.188282,0.282209,0.228507,0.180105
5,0.342500,2.734942,0.199561,0.337423,0.293064,0.194785


### wav2vec2-lg-xlsr-en-speech-emotion-recognition

In [ ]:
MODEL_NAME_AUDIO = "ehcalabres/wav2vec2-lg-xlsr-en-speech-emotion-recognition"  # AST

# -----------------------------
# DATALOADERS
# -----------------------------
fe = AutoFeatureExtractor.from_pretrained(MODEL_NAME_AUDIO)

train_ds = EmotionAudioDataset(df_train, fe)
val_ds   = EmotionAudioDataset(df_val, fe)
test_ds  = EmotionAudioDataset(df_test, fe)

collator = DataCollatorAudio(feature_extractor=fe)

# -----------------------------
# MODEL (head baru 8 kelas)
# -----------------------------
import torch.nn as nn

model_audio = AutoModelForAudioClassification.from_pretrained(
    MODEL_NAME_AUDIO,
    num_labels=NUM_CLASSES,
    label2id=label2id,
    id2label=id2label,
    problem_type="single_label_classification",
    ignore_mismatched_sizes=True,  # penting: ganti head
)

# -----------------------------
# CLASS WEIGHTS (imbalanced)
# -----------------------------
counts = np.bincount(df_train["emotion_label"].values, minlength=NUM_CLASSES)
# bobot ~ total/(K * count)
class_weights = (counts.sum() / (counts + 1e-6)) / NUM_CLASSES
class_weights = torch.tensor(class_weights, dtype=torch.float)

# -----------------------------
# METRICS
# -----------------------------
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = logits.argmax(-1)
    return {
        "macro_f1": f1_score(labels, preds, average="macro"),
        "accuracy": accuracy_score(labels, preds),
        "precision_macro": precision_score(labels, preds, average="macro", zero_division=0),
        "recall_macro": recall_score(labels, preds, average="macro", zero_division=0),
    }

# -----------------------------
# TRAINER dengan weighted CE
# -----------------------------
from transformers import Trainer

import torch.nn as nn
from transformers import Trainer

class WeightedTrainer(Trainer):
    def __init__(self, class_weights, **kwargs):
        super().__init__(**kwargs)
        self.class_weights = class_weights

    # Tambahkan num_items_in_batch dengan default None (atau **kwargs)
    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        # Jangan mutasi inputs dari luar secara permanen (copy ringan)
        labels = inputs.get("labels")
        # Buang labels saat forward
        _inputs = {k: v for k, v in inputs.items() if k != "labels"}

        outputs = model(**_inputs)
        logits = outputs.logits

        loss_fct = nn.CrossEntropyLoss(weight=self.class_weights.to(logits.device))
        # pastikan shape sesuai (N, num_labels)
        loss = loss_fct(logits.view(-1, self.model.config.num_labels),
                        labels.view(-1))

        return (loss, outputs) if return_outputs else loss

args_audio = TrainingArguments(
    output_dir="./training/wav2vec2_audio_2",
    overwrite_output_dir=False,
    learning_rate=1e-5,
    num_train_epochs=5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    dataloader_num_workers=4,
    fp16=True,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    greater_is_better=True,
    save_total_limit=2,
    report_to="none",
    seed=SEED,
)

trainer_audio = WeightedTrainer(
    class_weights=class_weights,
    model=model_audio,
    args=args_audio,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    data_collator=collator,
    compute_metrics=compute_metrics,
)

trainer_audio.train()
# Optional: simpan best
trainer_audio.save_model("./training/wav2vec2_audio_best")


Fetching 1 files:   0%|          | 0/1 [00:00<?, ?it/s]

preprocessor_config.json:   0%|          | 0.00/214 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.27G [00:00<?, ?B/s]

Some weights of the model checkpoint at ehcalabres/wav2vec2-lg-xlsr-en-speech-emotion-recognition were not used when initializing Wav2Vec2ForSequenceClassification: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.output.bias', 'classifier.output.weight']
- This IS expected if you are initializing Wav2Vec2ForSequenceClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing Wav2Vec2ForSequenceClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
Some weights of Wav2Vec2ForSequenceClassification were not initialized from the model checkpoint at ehcalabres/wav2vec2-lg-xlsr-en-speech-emotion-recognition and are newly initialized: ['classifier.bias', 'classifier.weight', '

Epoch,Training Loss,Validation Loss,Macro F1,Accuracy,Precision Macro,Recall Macro
1,2.072900,2.047061,0.093348,0.417178,0.208002,0.132460
2,2.039600,2.022462,0.085896,0.411043,0.207674,0.128553
3,2.016000,2.006328,0.096991,0.386503,0.139286,0.126157
4,2.000700,1.998232,0.094700,0.282209,0.085799,0.108724
5,1.996000,1.996047,0.092723,0.282209,0.084658,0.106684


In [ ]:
MODEL_NAME_AUDIO = "ehcalabres/wav2vec2-lg-xlsr-en-speech-emotion-recognition"  # AST

# -----------------------------
# DATALOADERS
# -----------------------------
fe = AutoFeatureExtractor.from_pretrained(MODEL_NAME_AUDIO)
# Pastikan sampling_rate feature extractor = 16000
if fe.sampling_rate != TARGET_SR:
    fe.sampling_rate = TARGET_SR  # supaya konsisten

train_ds = EmotionAudioDataset(df_train, fe)
val_ds   = EmotionAudioDataset(df_val, fe)
test_ds  = EmotionAudioDataset(df_test, fe)

collator = DataCollatorAudio(feature_extractor=fe)

# -----------------------------
# MODEL (head baru 8 kelas)
# -----------------------------
import torch.nn as nn

model_audio = AutoModelForAudioClassification.from_pretrained(
    MODEL_NAME_AUDIO,
    num_labels=NUM_CLASSES,
    label2id=id2label,
    id2label=id2label,
    problem_type="single_label_classification",
    ignore_mismatched_sizes=True,  # penting: ganti head
)

# -----------------------------
# CLASS WEIGHTS (imbalanced)
# -----------------------------
counts = np.bincount(df_train["emotion_label"].values, minlength=NUM_CLASSES)
# bobot ~ total/(K * count)
class_weights = (counts.sum() / (counts + 1e-6)) / NUM_CLASSES
class_weights = torch.tensor(class_weights, dtype=torch.float)

# -----------------------------
# METRICS
# -----------------------------
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = logits.argmax(-1)
    return {
        "macro_f1": f1_score(labels, preds, average="macro"),
        "accuracy": accuracy_score(labels, preds),
        "precision_macro": precision_score(labels, preds, average="macro", zero_division=0),
        "recall_macro": recall_score(labels, preds, average="macro", zero_division=0),
    }

# -----------------------------
# TRAINER dengan weighted CE
# -----------------------------
from transformers import Trainer

import torch.nn as nn
from transformers import Trainer

class WeightedTrainer(Trainer):
    def __init__(self, class_weights, **kwargs):
        super().__init__(**kwargs)
        self.class_weights = class_weights

    # Tambahkan num_items_in_batch dengan default None (atau **kwargs)
    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        # Jangan mutasi inputs dari luar secara permanen (copy ringan)
        labels = inputs.get("labels")
        # Buang labels saat forward
        _inputs = {k: v for k, v in inputs.items() if k != "labels"}

        outputs = model(**_inputs)
        logits = outputs.logits

        loss_fct = nn.CrossEntropyLoss(weight=self.class_weights.to(logits.device))
        # pastikan shape sesuai (N, num_labels)
        loss = loss_fct(logits.view(-1, self.model.config.num_labels),
                        labels.view(-1))

        return (loss, outputs) if return_outputs else loss

args_audio = TrainingArguments(
    output_dir="./training/wav2vec2_audio_2",
    overwrite_output_dir=False,
    learning_rate=5e-6,
    num_train_epochs=5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    dataloader_num_workers=4,
    fp16=True,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    greater_is_better=True,
    save_total_limit=2,
    report_to="none",
    seed=SEED,
)

trainer_audio = WeightedTrainer(
    class_weights=class_weights,
    model=model_audio,
    args=args_audio,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    data_collator=collator,
    compute_metrics=compute_metrics,
)

trainer_audio.train()
# Optional: simpan best
trainer_audio.save_model("./training/wav2vec2_audio_best_2")


Fetching 1 files:   0%|          | 0/1 [00:00<?, ?it/s]

Some weights of the model checkpoint at ehcalabres/wav2vec2-lg-xlsr-en-speech-emotion-recognition were not used when initializing Wav2Vec2ForSequenceClassification: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.output.bias', 'classifier.output.weight']
- This IS expected if you are initializing Wav2Vec2ForSequenceClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing Wav2Vec2ForSequenceClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
Some weights of Wav2Vec2ForSequenceClassification were not initialized from the model checkpoint at ehcalabres/wav2vec2-lg-xlsr-en-speech-emotion-recognition and are newly initialized: ['classifier.bias', 'classifier.weight', '

Epoch,Training Loss,Validation Loss,Macro F1,Accuracy,Precision Macro,Recall Macro
1,2.060000,2.033794,0.090421,0.257669,0.115311,0.128348
2,2.039200,2.017659,0.111515,0.331288,0.121735,0.130543
3,2.023700,2.008388,0.101463,0.319018,0.106250,0.121746
4,2.006300,2.003264,0.102320,0.325153,0.102710,0.122099
5,2.006000,2.002326,0.096791,0.312883,0.097049,0.115342


In [ ]:
MODEL_NAME_AUDIO = "audeering/wav2vec2-large-robust-12-ft-emotion-msp-dim"

# -----------------------------
# DATALOADERS
# -----------------------------
fe = AutoFeatureExtractor.from_pretrained(MODEL_NAME_AUDIO)

train_ds = EmotionAudioDataset(df_train, fe)
val_ds   = EmotionAudioDataset(df_val, fe)
test_ds  = EmotionAudioDataset(df_test, fe)

collator = DataCollatorAudio(feature_extractor=fe)

# -----------------------------
# MODEL (head baru 8 kelas)
# -----------------------------
import torch.nn as nn

model_audio = AutoModelForAudioClassification.from_pretrained(
    MODEL_NAME_AUDIO,
    num_labels=NUM_CLASSES,
    label2id=label2id,
    id2label=id2label,
    problem_type="single_label_classification",
    ignore_mismatched_sizes=True,  # penting: ganti head
)

# -----------------------------
# CLASS WEIGHTS (imbalanced)
# -----------------------------
counts = np.bincount(df_train["emotion_label"].values, minlength=NUM_CLASSES)
# bobot ~ total/(K * count)
class_weights = (counts.sum() / (counts + 1e-6)) / NUM_CLASSES
class_weights = torch.tensor(class_weights, dtype=torch.float)

# -----------------------------
# METRICS
# -----------------------------
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = logits.argmax(-1)
    return {
        "macro_f1": f1_score(labels, preds, average="macro"),
        "accuracy": accuracy_score(labels, preds),
        "precision_macro": precision_score(labels, preds, average="macro", zero_division=0),
        "recall_macro": recall_score(labels, preds, average="macro", zero_division=0),
    }

# -----------------------------
# TRAINER dengan weighted CE
# -----------------------------
from transformers import Trainer

import torch.nn as nn
from transformers import Trainer

class WeightedTrainer(Trainer):
    def __init__(self, class_weights, **kwargs):
        super().__init__(**kwargs)
        self.class_weights = class_weights

    # Tambahkan num_items_in_batch dengan default None (atau **kwargs)
    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        # Jangan mutasi inputs dari luar secara permanen (copy ringan)
        labels = inputs.get("labels")
        # Buang labels saat forward
        _inputs = {k: v for k, v in inputs.items() if k != "labels"}

        outputs = model(**_inputs)
        logits = outputs.logits

        loss_fct = nn.CrossEntropyLoss(weight=self.class_weights.to(logits.device))
        # pastikan shape sesuai (N, num_labels)
        loss = loss_fct(logits.view(-1, self.model.config.num_labels),
                        labels.view(-1))

        return (loss, outputs) if return_outputs else loss

args_audio = TrainingArguments(
    output_dir="./training/clap-htsat_audio",
    overwrite_output_dir=False,
    learning_rate=1e-5,
    num_train_epochs=5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    dataloader_num_workers=4,
    fp16=True,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    greater_is_better=True,
    save_total_limit=2,
    report_to="none",
    seed=SEED,
)

trainer_audio = WeightedTrainer(
    class_weights=class_weights,
    model=model_audio,
    args=args_audio,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    data_collator=collator,
    compute_metrics=compute_metrics,
)

trainer_audio.train()
# Optional: simpan best
trainer_audio.save_model("./training/clap-htsat_audio_best")


Fetching 1 files:   0%|          | 0/1 [00:00<?, ?it/s]

preprocessor_config.json:   0%|          | 0.00/214 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/661M [00:00<?, ?B/s]

Some weights of Wav2Vec2ForSequenceClassification were not initialized from the model checkpoint at audeering/wav2vec2-large-robust-12-ft-emotion-msp-dim and are newly initialized: ['classifier.bias', 'classifier.weight', 'projector.bias', 'projector.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Epoch,Training Loss,Validation Loss,Macro F1,Accuracy,Precision Macro,Recall Macro
1,2.077200,2.071527,0.134290,0.300613,0.189501,0.149674
2,2.071900,2.067282,0.148176,0.398773,0.214941,0.167986
3,2.067900,2.064100,0.141474,0.404908,0.208549,0.158841
4,2.063900,2.061579,0.134254,0.386503,0.181417,0.151731
5,2.063000,2.060924,0.127012,0.386503,0.195034,0.148706


In [ ]:
SEED = 42

In [ ]:
import os
import pandas as pd
import numpy as np
import torch
import torchaudio
from dataclasses import dataclass, field
from typing import Dict, List, Any

from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, accuracy_score

from transformers import (
    AutoFeatureExtractor,
    ASTForAudioClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding,
)

# Hapus warning yang tidak relevan
import warnings
warnings.filterwarnings("ignore")

print("✅ Pustaka berhasil diimpor.")

# =================================================================
# 1. KONFIGURASI ⚙️
# =================================================================
@dataclass
class Config:
    NUM_CLASSES: int = 8
    SEED: int = 42
    MODEL_NAME_AUDIO: str = "MIT/ast-finetuned-audioset-10-10-0.4593"
    TARGET_SR: int = 16000
    MAX_SECONDS: float = 10.0 # Kita akan potong atau pad semua audio ke durasi ini
    MAX_LEN_AUDIO: int = field(init=False)

    OUTPUT_DIR: str = "./training/audio_only_model"
    EPOCHS: int = 5 # Kita bisa tambah epoch karena model lebih ringan
    TRAIN_BS: int = 2 # Naikkan batch size, model audio saja lebih ringan
    EVAL_BS: int = 2
    LR: float = 3e-5

    def __post_init__(self):
        self.MAX_LEN_AUDIO = int(self.TARGET_SR * self.MAX_SECONDS)

config = Config()

label2id = {
    'Proud': 0, 'Trust': 1, 'Joy': 2, 'Surprise': 3,
    'Neutral': 4, 'Sadness': 5, 'Fear': 6, 'Anger': 7
}
id2label = {v: k for k, v in label2id.items()}

print("✅ Konfigurasi selesai.")

# --- Ganti baris ini dengan `pd.read_csv(...)` Anda ---
df_all = pd.read_csv("./data/df_train_processed.csv")

# Split stratified by label (konsisten dengan video)
df_train, df_temp = train_test_split(
    df_all, test_size=0.3, random_state=SEED, stratify=df_all["emotion_label"]
)
df_val, df_test = train_test_split(
    df_temp, test_size=0.3, random_state=SEED, stratify=df_temp["emotion_label"]
)


print(f"Total data: {len(df_all)}, Train: {len(df_train)}, Val: {len(df_val)}")

# =================================================================
# 3. CUSTOM DATASET AUDIO 🎵
# =================================================================
class EmotionAudioDataset(torch.utils.data.Dataset):
    def __init__(self, df, feature_extractor, config):
        self.df = df.reset_index(drop=True)
        self.feature_extractor = feature_extractor
        self.config = config

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        audio_path = row["audio_path"]

        try:
            wav, sr = torchaudio.load(audio_path)
            if wav.shape[0] > 1: wav = wav.mean(dim=0, keepdim=True)
            if sr != self.config.TARGET_SR:
                wav = torchaudio.functional.resample(wav, sr, self.config.TARGET_SR)
            wav = wav.squeeze(0)

            # Center-crop atau pad ke panjang tetap
            n = wav.shape[0]
            if n > self.config.MAX_LEN_AUDIO:
                start = (n - self.config.MAX_LEN_AUDIO) // 2
                wav = wav[start:start + self.config.MAX_LEN_AUDIO]
            elif n < self.config.MAX_LEN_AUDIO:
                wav = torch.nn.functional.pad(wav, (0, self.config.MAX_LEN_AUDIO - n))

            # Ekstrak fitur (spektrogram) di sini
            inputs = self.feature_extractor(
                wav.numpy(), sampling_rate=self.config.TARGET_SR, return_tensors="pt"
            )
            # Dapatkan input_values, hapus dimensi batch yang tidak perlu
            input_values = inputs.input_values.squeeze(0)

        except Exception as e:
            print(f"⚠️ Gagal memuat {audio_path}: {e}. Menggunakan data sunyi.")
            input_values = torch.zeros((128, 1024), dtype=torch.float32) # Shape dummy untuk AST

        return {
            "input_values": input_values,
            "labels": torch.tensor(row["emotion_label"], dtype=torch.long)
        }

# =================================================================
# 4. INISIALISASI & TRAINING 🚀
# =================================================================
# Inisialisasi feature extractor dan model
feature_extractor = AutoFeatureExtractor.from_pretrained(config.MODEL_NAME_AUDIO)
model = ASTForAudioClassification.from_pretrained(
    config.MODEL_NAME_AUDIO,
    num_labels=config.NUM_CLASSES,
    label2id=label2id,
    id2label=id2label,
    ignore_mismatched_sizes=True,
)

# Buat dataset
train_ds = EmotionAudioDataset(df_train, feature_extractor, config)
val_ds = EmotionAudioDataset(df_val, feature_extractor, config)

# Data collator
# Kita bisa gunakan DataCollatorWithPadding karena dataset sudah menghasilkan tensor dengan shape yang sama
data_collator = DataCollatorWithPadding(tokenizer=feature_extractor, padding=True)

# Metrik evaluasi
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {"macro_f1": f1_score(labels, preds, average="macro"), "accuracy": accuracy_score(labels, preds)}

# Argumen training
training_args = TrainingArguments(
    output_dir=config.OUTPUT_DIR,
    overwrite_output_dir=True,
    num_train_epochs=config.EPOCHS,
    per_device_train_batch_size=config.TRAIN_BS,
    per_device_eval_batch_size=config.EVAL_BS,
    learning_rate=config.LR,
    fp16=torch.cuda.is_available(),
    logging_strategy="epoch",
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    greater_is_better=True,
    save_total_limit=1,
    report_to="none",

)

# Buat Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

# Mulai training
if df_all is not None:
    print("\n\n--- MEMULAI TRAINING AUDIO ---")
    trainer.train()
    print("--- TRAINING AUDIO SELESAI ---\n\n")
    trainer.save_model(os.path.join(config.OUTPUT_DIR, "best_model"))
    print(f"✅ Model audio terbaik disimpan di: {os.path.join(config.OUTPUT_DIR, 'best_model')}")

✅ Pustaka berhasil diimpor.
✅ Konfigurasi selesai.
Total data: 803, Train: 562, Val: 168


Fetching 1 files:   0%|          | 0/1 [00:00<?, ?it/s]

Some weights of ASTForAudioClassification were not initialized from the model checkpoint at MIT/ast-finetuned-audioset-10-10-0.4593 and are newly initialized because the shapes did not match:
- classifier.dense.bias: found shape torch.Size([527]) in the checkpoint and torch.Size([8]) in the model instantiated
- classifier.dense.weight: found shape torch.Size([527, 768]) in the checkpoint and torch.Size([8, 768]) in the model instantiated
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


OutOfMemoryError: CUDA out of memory. Tried to allocate 2.00 MiB. GPU 0 has a total capacity of 14.74 GiB of which 2.12 MiB is free. Process 3301 has 14.74 GiB memory in use. Of the allocated memory 14.19 GiB is allocated by PyTorch, and 427.46 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)